# 01 — Data Ingestion from TDMS

Reads raw DAS fiber-optic sensor data from AAR/TTCI TDMS files.

**Dataset:** HTL loop, 4.16 km fiber-optic railroad track, Pueblo CO  
**Format:** National Instruments TDMS — multi-channel time-series, ~310,000 rows × 4,160 channels  
**Classes:** NC (Normal Condition), TP (Train Position), AC1 (Anomaly Class 1), AC2 (Anomaly Class 2)  

**Paper:** Rahman et al., *Green Energy and Intelligent Transportation*, Elsevier 2024

## Install dependencies
```bash
pip install nptdms pandas numpy matplotlib
```

In [ ]:
import os
import numpy as np
import pandas as pd
from nptdms import TdmsFile

# ── Load a single TDMS file ────────────────────────────────────────────────
# Replace with the path to your TDMS file
TDMS_FILE_PATH = "data/raw/your_das_file.tdms"

tdms_file = TdmsFile(TDMS_FILE_PATH)

data = {}
for channel in tdms_file.groups()[0].channels():
    data[channel.name] = channel.data

df = pd.DataFrame(data)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]:,} channels")
df.head()

## Batch Loading — Multiple TDMS Files

In [ ]:
TDMS_DIR = "data/raw/"

all_frames = []
for fname in sorted(os.listdir(TDMS_DIR)):
    if not fname.endswith(".tdms"):
        continue
    tdms_file = TdmsFile(os.path.join(TDMS_DIR, fname))
    data = {ch.name: ch.data for ch in tdms_file.groups()[0].channels()}
    frame = pd.DataFrame(data)
    frame["source_file"] = fname
    all_frames.append(frame)
    print(f"  Loaded {fname}: {frame.shape}")

df_full = pd.concat(all_frames, ignore_index=True)
print(f"Full dataset: {df_full.shape[0]:,} rows x {df_full.shape[1]:,} columns")

## Save to CSV for downstream feature extraction

In [ ]:
os.makedirs("data/processed", exist_ok=True)
df_full.to_csv("data/processed/raw_das_signal.csv", index=False)
print("Saved to data/processed/raw_das_signal.csv")

## Raw Signal Visualisation

In [ ]:
import matplotlib.pyplot as plt

channel = df.columns[0]
plt.figure(figsize=(14, 3))
plt.plot(df[channel].values, linewidth=0.4)
plt.title(f"Rail Fiber-Optic Signal — Channel: {channel}", fontweight="bold")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.savefig("results/figures/raw_signal.png", dpi=150, bbox_inches="tight")
plt.show()